# Exercise 12: bringing it all together — a promoter benchmark

Predict whether a 251-base human DNA sequence is a non-TATA promoter: **1 = promoter, 0 = non-promoter**. This notebook downloads its own data and compares six complete prediction paths on exactly the same examples.

| Path | Representation | Classifier |
| --- | --- | --- |
| k-mer + logistic regression | Overlapping 6-mer counts | One sigmoid output unit |
| k-mer + random forest | The same counts | Bagged decision trees |
| k-mer + XGBoost | The same counts | Gradient-boosted decision trees |
| Sequence CNN | One-hot DNA in sequence order | Two learned convolutional layers |
| Transformer + logistic regression | Mean frozen token embedding | Scaled features + one sigmoid output unit |
| Transformer + CNN | Ordered frozen token embeddings | One learned convolutional layer + output unit |

Here **perceptron means the requested sigmoid linear classifier, implemented as logistic regression**. The classical hard-threshold perceptron uses a different training rule and does not directly produce probabilities. We use logistic regression so probability metrics can be compared across all six paths.

The questions are: which representation/classifier works best under these settings, how reliable are its probabilities, what does it cost, and do different models make different mistakes? A single benchmark cannot establish a universally best model or a biological promoter mechanism. **Run all includes the official test evaluation.**


## Setup and benchmark protocol

A Colab T4 GPU is recommended for the transformer and CNNs; CPU execution is supported but slower. Trees and logistic regression run on CPU. Install the libraries below (`%pip` targets this notebook's environment, `-q` reduces messages). **XGBoost is the actual `xgboost` library**, using its CPU histogram tree algorithm. Pinned versions make model loading and the boosting API reproducible. Local users can install the repository's `requirements.txt`.

We keep the official test split untouched until all models are fitted. Reserve 20% per class from official training for validation. All settings below are declared in advance: there is no hyperparameter search. Validation selects the two CNN epochs and XGBoost's boosting round. Validation ROC AUC nominates one model before the test is opened; all six predeclared models are then reported. This is a comparison of these settings, not an equal tuning-budget competition.

The embedding function and token CNN use the same implementation as Exercise 11, included here so this notebook runs on its own. Token features are cached in CPU RAM with 16-bit precision (roughly 2 GB total); training batches use 32-bit arithmetic. Benchmark tables, plots and model artifacts are saved in `exercise_12_output`. Installation, downloads, loading pretrained weights and feature extraction are separated from downstream fitting times. The pretrained model's original training cost is unknown and is not counted.


In [ ]:
%pip install -q pyarrow transformers==4.57.1 xgboost==3.1.1


### Read the settings before running

`seed=42` fixes the split, initialization and randomized algorithms; it does not guarantee bitwise agreement across hardware. `k=6` gives at most 4⁶ = 4,096 known-base count features. `threshold=0.5` is the same decision rule for every model. Set any alternative threshold using validation data only.

`batch_size=32` limits transformer inference memory. `cnn_batch_size=128` is larger because the downstream CNNs are small. `max_epochs=15` bounds each CNN's training passes; validation loss chooses its checkpoint. Adam's `learning_rate=0.001` is a conventional starting value. `n_jobs=2` and two PyTorch threads limit CPU resource use consistently. `bootstrap_repeats=500` balances confidence-interval stability against runtime. Model/data revisions below pin the inputs.

`clock_time()` synchronizes CUDA before reading wall time, so asynchronous GPU work is included. These are single-run measurements on the printed machine; they are not hardware-independent speed rankings.


In [ ]:
from copy import deepcopy
from pathlib import Path
from time import perf_counter
import json
import os
import platform
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import transformers
from transformers import AutoTokenizer, AutoModelForMaskedLM
import xgboost
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, log_loss, matthews_corrcoef, brier_score_loss,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay,
)

seed = 42
k = 6
threshold = 0.5
batch_size = 32
cnn_batch_size = 128
max_epochs = 15
learning_rate = 0.001
n_jobs = 2
bootstrap_repeats = 500
model_name = 'InstaDeepAI/nucleotide-transformer-v2-50m-multi-species'
model_revision = '81b29e5786726d891dbf929404ef20adca5b36f1'
output_dir = Path('exercise_12_output')
output_dir.mkdir(exist_ok=True)
torch.manual_seed(seed)
torch.set_num_threads(n_jobs)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def clock_time():
    if device.type == 'cuda':
        torch.cuda.synchronize()
    return perf_counter()

models = {}
fit_seconds = {}
histories = {}
selected_steps = {}
print('Device:', device, '| CPU threads:', n_jobs)
print('Python:', platform.python_version(), '| PyTorch:', torch.__version__)
print('scikit-learn:', sklearn.__version__, '| XGBoost:', xgboost.__version__)


## 1. One dataset, one split, one row order

Use the [Genomic Benchmarks human non-TATA promoter dataset](https://huggingface.co/datasets/katarinagresova/Genomic_Benchmarks_human_nontata_promoters) at a fixed revision. `read_parquet` reads its compressed tables. Check every sequence against `[ACGTN]{251}`: 251 allowed bases, including unknown base `N`. Keep all rows.

`groupby('label').sample(frac=0.20, random_state=seed)` reserves 20% of each label for validation; `drop` removes those exact row indices from training. This gives **21,678 training / 5,419 validation / 9,034 official test examples**. Preserve the original indices in exported predictions. Each representation follows the DataFrame's row order, and only training batches are shuffled. This is an existing benchmark split, not a holdout of entire chromosomes or unseen species.


In [ ]:
data_url = (
    'https://huggingface.co/datasets/katarinagresova/'
    'Genomic_Benchmarks_human_nontata_promoters/resolve/'
    'e0003669df0a180c8570ebe091ed91f20fa080ec/data/'
)
started = clock_time()
benchmark_train = pd.read_parquet(data_url + 'train-00000-of-00001-aeb55728756a44ae.parquet')
benchmark_train = benchmark_train.rename(columns={'seq': 'sequence'})
training_data_download_seconds = clock_time() - started
assert benchmark_train['sequence'].str.fullmatch('[ACGTN]{251}').all()
assert set(benchmark_train['label']) == {0, 1}
validation_data = benchmark_train.groupby('label').sample(frac=0.20, random_state=seed)
train_data = benchmark_train.drop(validation_data.index)
y_train = train_data['label'].to_numpy()
y_validation = validation_data['label'].to_numpy()
split_counts = pd.DataFrame({
    'train': train_data['label'].value_counts(),
    'validation': validation_data['label'].value_counts(),
}).T
print(split_counts.rename(columns={0: 'Non-promoter', 1: 'Promoter'}))
print('Training:', len(train_data), '| Validation:', len(validation_data))


## 2. Shared 6-mer counts and three classical classifiers

`analyzer='char'` counts overlapping strings of bases; `ngram_range=(6,6)` selects only 6-mers. A known-base sequence has 246 windows. `lowercase=False` preserves the uppercase DNA alphabet, and `dtype=float32` saves memory and suits XGBoost. Fit the vocabulary on training sequences only and transform validation/test without refitting. An unseen k-mer containing `N` is ignored, while its sequence stays in the benchmark. The sparse matrix stores nonzero counts efficiently.

- **Logistic regression:** `C=1.0` is inverse L2 regularization strength (smaller means stronger penalty); `solver='liblinear'` handles this binary sparse problem; `max_iter=1000` limits optimizer iterations. All count features have the same units, so we use raw counts for this baseline.
- **Random forest:** 200 trees average predictions from bootstrap samples. `max_features='sqrt'` tests a random square-root-sized feature subset at each split. `min_samples_leaf=2` discourages leaves based on one training example. Trees have no explicit depth cap. `n_jobs=2` limits CPU workers.
- **XGBoost:** at most 400 sequential trees, each of depth at most 4. `learning_rate=0.05` reduces each tree's contribution; `subsample=0.8` and `colsample_bytree=0.8` randomly sample training rows and features to reduce overfitting. `reg_lambda=1.0` penalizes large leaf weights. `objective='binary:logistic'` produces class-1 probabilities. `tree_method='hist'` uses histogram splits on CPU. Validation `logloss` with `early_stopping_rounds=25` stops after 25 rounds without improvement and prediction uses the best round.

XGBoost treats implicit sparse zeros as missing values. To make zero counts explicit numerical values, we give it a dense copy and measure the conversion cost. These copies are released after fitting. The other two models use the same count values in sparse format. No test data enter any fit or stopping decision.


In [ ]:
vectorizer = CountVectorizer(analyzer='char', ngram_range=(k, k), lowercase=False, dtype=np.float32)
started = clock_time()
train_counts = vectorizer.fit_transform(train_data['sequence'])
kmer_train_seconds = clock_time() - started
started = clock_time()
validation_counts = vectorizer.transform(validation_data['sequence'])
kmer_validation_seconds = clock_time() - started
models['k-mer + logistic regression'] = LogisticRegression(
    C=1.0, solver='liblinear', max_iter=1000, random_state=seed,
)
models['k-mer + random forest'] = RandomForestClassifier(
    n_estimators=200, max_features='sqrt', min_samples_leaf=2, random_state=seed, n_jobs=n_jobs,
)
for name, model in models.items():
    started = clock_time()
    model.fit(train_counts, y_train)
    fit_seconds[name] = clock_time() - started
    print(name, 'fit seconds:', round(fit_seconds[name], 2), flush=True)

started = clock_time()
train_dense = train_counts.toarray()
xgb_train_dense_seconds = clock_time() - started
started = clock_time()
validation_dense = validation_counts.toarray()
xgb_validation_dense_seconds = clock_time() - started
name = 'k-mer + XGBoost'
models[name] = XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.0, objective='binary:logistic', eval_metric='logloss', tree_method='hist',
    early_stopping_rounds=25, random_state=seed, n_jobs=n_jobs,
)
started = clock_time()
models[name].fit(train_dense, y_train, eval_set=[(validation_dense, y_validation)], verbose=False)
fit_seconds[name] = clock_time() - started
selected_steps[name] = models[name].best_iteration + 1
del train_dense, validation_dense
print('Count features:', train_counts.shape[1])
print(name, 'selected trees:', selected_steps[name], '| fit seconds:', round(fit_seconds[name], 2), flush=True)


## 3. A short training loop for the two neural classifiers

Both CNNs use `BCEWithLogitsLoss`: it includes sigmoid inside a numerically stable binary cross-entropy calculation. Models output logits, and sigmoid is applied only for reported probabilities. Adam updates only the classifier's parameters, with learning rate 0.001 and `weight_decay=0.0001` as a small penalty on large weights.

The loop accepts either one input tensor (one-hot DNA) or two (token vectors and mask). `move_features` converts floating inputs to 32-bit and preserves Boolean masks. `zero_grad`, `backward` and `step` clear gradients, compute new gradients and update weights. Training loaders shuffle; evaluation loaders preserve row order. Labels use floating-point 0/1 for the loss.

Each epoch measures training loss with dropout and changing weights, then validation loss with `eval()` and `no_grad()`. Validation makes no updates. A deep copy saves the best weights independently of later training. Restore the lowest-validation-loss epoch after 15 passes. These selection passes are included in fit time. Equal epochs do not imply equal compute or model capacity.


In [ ]:
def move_features(features):
    return [value.to(device, dtype=torch.float32) if value.is_floating_point()
            else value.to(device) for value in features]

def train_network(model, train_loader, validation_loader, name):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=0.0001)
    loss_function = nn.BCEWithLogitsLoss()
    history = []
    best_loss = np.inf
    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0.0
        for *features, labels in train_loader:
            optimizer.zero_grad()
            logits = model(*move_features(features)).reshape(-1)
            loss = loss_function(logits, labels.to(device))
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
        training_loss = total_loss / len(train_loader.dataset)
        model.eval()
        total_loss = 0.0
        with torch.no_grad():
            for *features, labels in validation_loader:
                logits = model(*move_features(features)).reshape(-1)
                total_loss += loss_function(logits, labels.to(device)).item() * len(labels)
        validation_loss = total_loss / len(validation_loader.dataset)
        history.append({'epoch': epoch, 'training_loss': training_loss, 'validation_loss': validation_loss})
        if validation_loss < best_loss:
            best_loss, best_epoch = validation_loss, epoch
            best_state = deepcopy(model.state_dict())
        print(f'{name} epoch {epoch}: train={training_loss:.4f}, validation={validation_loss:.4f}', flush=True)
    model.load_state_dict(best_state)
    model.eval()
    return pd.DataFrame(history), best_epoch

def predict_network(model, loader):
    model.eval()
    batches = []
    with torch.no_grad():
        for *features, _ in loader:
            logits = model(*move_features(features)).reshape(-1)
            batches.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(batches)


## 4. Sequence → learned CNN

Encode A/C/G/T as four indicator channels and `N` as four zeros. This is a fixed mapping with no vocabulary to fit. Shape is `(examples, 4, 251)`; base order is retained.

The first convolution learns 32 filters of width 15 bases. `padding=7` preserves length; ReLU adds nonlinearity. Width-2 max pooling halves resolution. A second convolution combines nearby responses with 64 filters of width 7 (on the reduced grid), and `padding=3` preserves that grid's length. `AdaptiveMaxPool1d(1)` retains the strongest response per channel across the sequence; flatten produces 64 features. `Dropout(0.3)` randomly drops 30% during training, then a linear output supplies one logit. All convolutional and output weights start randomly and learn from promoter labels. Filters need not become recognizable biological motifs.


In [ ]:
def encode_sequences(sequences):
    base_ids = {'A': 0, 'C': 1, 'G': 2, 'T': 3, 'N': 4}
    ids = np.array([[base_ids[base] for base in sequence] for sequence in sequences])
    vectors = np.vstack([np.eye(4), np.zeros((1, 4))]).astype(np.float32)
    return torch.from_numpy(vectors[ids]).permute(0, 2, 1).contiguous()

started = clock_time()
train_onehot = encode_sequences(train_data['sequence'])
raw_train_seconds = clock_time() - started
started = clock_time()
validation_onehot = encode_sequences(validation_data['sequence'])
raw_validation_seconds = clock_time() - started
raw_train_loader = DataLoader(TensorDataset(train_onehot, torch.tensor(y_train, dtype=torch.float32)),
                              batch_size=cnn_batch_size, shuffle=True)
raw_validation_loader = DataLoader(TensorDataset(validation_onehot, torch.tensor(y_validation, dtype=torch.float32)),
                                   batch_size=cnn_batch_size, shuffle=False)
name = 'Sequence CNN'
models[name] = nn.Sequential(
    nn.Conv1d(4, 32, kernel_size=15, padding=7), nn.ReLU(), nn.MaxPool1d(2),
    nn.Conv1d(32, 64, kernel_size=7, padding=3), nn.ReLU(),
    nn.AdaptiveMaxPool1d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, 1),
).to(device)
started = clock_time()
histories[name], selected_steps[name] = train_network(models[name], raw_train_loader, raw_validation_loader, name)
fit_seconds[name] = clock_time() - started


## 5. Sequence → frozen transformer features

Load the [50M Nucleotide Transformer v2](https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-50m-multi-species), pretrained to predict masked DNA tokens in 850 species. The embedding workflow is adapted from Andrea Gariboldi's [bioFoundationModels notebook](https://github.com/andrea-gariboldi/bioFoundationModels/blob/main/bioFoundation_models.ipynb). This model has learned from substantial external data; its downstream advantage would not isolate architecture alone.

`trust_remote_code=True` loads the checkpoint's custom Python architecture; the exact `revision` pins that code and its weights. `requires_grad_(False)` freezes weights, `eval()` selects inference behavior, and `inference_mode()` avoids gradient bookkeeping. They serve different purposes. Neither promoter classifier fine-tunes the transformer.

The tokenizer uses non-overlapping 6-mers where possible, single bases for remainders and `N`, and adds `<cls>`. These tokens differ from overlapping count features. `padding=True` makes equal-width tensors within each split, `truncation=False` keeps all bases, and the attention mask excludes padding from attention. A DNA mask also excludes special tokens from our features. `hidden_states[-1]` selects the last layer's 512 features per token; vocabulary logits are not promoter probabilities.

Average DNA token vectors for logistic regression, giving `(examples, 512)`. Each token has equal weight, including single-base tokens. Keep ordered vectors for the CNN, giving `(examples, tokens, 512)`, and zero excluded positions. `torch.empty` reserves CPU storage filled batch by batch. Token caches use `float16` to halve RAM use, so the CNN sees slightly rounded embeddings. Mean vectors use `float32`. The embedding function is copied directly from Exercise 11. The two paths share one inference pass per batch; shared extraction costs are charged once per standalone path in the comparison, and must not be summed as actual joint runtime.


In [ ]:
started = clock_time()
tokenizer = AutoTokenizer.from_pretrained(model_name, revision=model_revision, trust_remote_code=True)
foundation_model = AutoModelForMaskedLM.from_pretrained(
    model_name, revision=model_revision, trust_remote_code=True,
).to(device)
foundation_model.requires_grad_(False)
foundation_model.eval()
model_load_seconds = clock_time() - started

def embed_sequences(sequences):
    sequences = list(sequences)
    inputs = tokenizer(sequences, padding=True, truncation=False, return_tensors='pt')
    special_ids = torch.tensor(tokenizer.all_special_ids)
    special_tokens = torch.isin(inputs['input_ids'], special_ids)
    dna_mask = inputs['attention_mask'].bool() & ~special_tokens
    token_features = torch.empty(
        len(sequences), inputs['input_ids'].shape[1],
        foundation_model.config.hidden_size, dtype=torch.float16,
    )
    batches = []
    with torch.inference_mode():
        for start in range(0, len(sequences), batch_size):
            stop = min(start + batch_size, len(sequences))
            batch_inputs = {name: values[start:stop].to(device) for name, values in inputs.items()}
            outputs = foundation_model(**batch_inputs, output_hidden_states=True)
            token_embeddings = outputs.hidden_states[-1]
            mask = dna_mask[start:stop].to(device).unsqueeze(-1)
            weights = mask.to(token_embeddings.dtype)
            pooled = (token_embeddings * weights).sum(dim=1) / weights.sum(dim=1)
            batches.append(pooled.cpu().numpy())
            token_features[start:stop] = token_embeddings.masked_fill(~mask, 0).cpu().half()
            print(f'Embedded {min(start + batch_size, len(sequences))}/{len(sequences)}',
                  end='\r', flush=True)
    print()
    return np.concatenate(batches).astype(np.float32), token_features, dna_mask

started = clock_time()
train_means, train_tokens, train_mask = embed_sequences(train_data['sequence'])
transformer_train_seconds = clock_time() - started
started = clock_time()
validation_means, validation_tokens, validation_mask = embed_sequences(validation_data['sequence'])
transformer_validation_seconds = clock_time() - started
print('Token cache shape:', tuple(train_tokens.shape), '| mean shape:', train_means.shape)


## 6. Two classifiers on the frozen features

**Mean + logistic regression:** `StandardScaler` learns a mean and standard deviation per feature from training data, then reuses them for all predictions. Scaling makes L2 regularization more comparable across embedding dimensions. A pipeline keeps scaling and classification together. `C=1.0` is a fixed baseline; `max_iter=2000` gives the default `lbfgs` solver enough optimization iterations.

**Token CNN:** transpose from `(batch, tokens, features)` to `(batch, channels, length)` for `Conv1d`. Learn 32 filters over three neighboring tokens and all 512 channels. `padding=0` uses complete windows only. Three tokens often span 18 bases, but single-base tokens change the span; contextual embeddings already incorporate information outside that window. ReLU adds nonlinearity, global max pooling retains the strongest response per filter, dropout 0.3 regularizes, and a linear layer outputs one logit.

A zeroed `<cls>` or padding vector could still affect convolutional responses through neighboring vectors or biases. `mask.unfold(1,3,1).all(-1)` marks only windows containing three DNA positions. Set all other responses to negative infinity so they cannot win max pooling. `N` remains a valid DNA position. This task has enough DNA tokens for valid windows in every example. Both mean and max pooling discard absolute position, and CNN filters on contextual vectors need not correspond to raw DNA motifs.


In [ ]:
name = 'Transformer + logistic regression'
models[name] = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000, random_state=seed))
started = clock_time()
models[name].fit(train_means, y_train)
fit_seconds[name] = clock_time() - started

class TokenCNN(nn.Module):
    def __init__(self, embedding_size):
        super().__init__()
        self.convolution = nn.Conv1d(embedding_size, 32, kernel_size=3, padding=0)
        self.dropout = nn.Dropout(0.3)
        self.output = nn.Linear(32, 1)

    def forward(self, tokens, mask):
        responses = torch.relu(self.convolution(tokens.transpose(1, 2)))
        valid_windows = mask.unfold(1, 3, 1).all(dim=-1)
        responses = responses.masked_fill(~valid_windows.unsqueeze(1), -torch.inf)
        pooled = responses.amax(dim=2)
        return self.output(self.dropout(pooled)).squeeze(1)

token_train_loader = DataLoader(
    TensorDataset(train_tokens, train_mask, torch.tensor(y_train, dtype=torch.float32)),
    batch_size=cnn_batch_size, shuffle=True,
)
token_validation_loader = DataLoader(
    TensorDataset(validation_tokens, validation_mask, torch.tensor(y_validation, dtype=torch.float32)),
    batch_size=cnn_batch_size, shuffle=False,
)
name = 'Transformer + CNN'
models[name] = TokenCNN(foundation_model.config.hidden_size).to(device)
started = clock_time()
histories[name], selected_steps[name] = train_network(models[name], token_train_loader, token_validation_loader, name)
fit_seconds[name] = clock_time() - started
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, history) in zip(axes, histories.items()):
    ax.plot(history['epoch'], history['training_loss'], label='Training (dropout)')
    ax.plot(history['epoch'], history['validation_loss'], label='Validation')
    ax.axvline(selected_steps[name], color='black', linestyle='--', label='Selected epoch')
    ax.set(title=name, xlabel='Epoch', ylabel='Binary cross-entropy')
    ax.legend()
fig.tight_layout()
fig.savefig(output_dir / 'learning_curves.png', dpi=150)
plt.show()


## 7. Validation comparison and a declared model choice

All classifiers predict a probability for promoter class 1. Accuracy, balanced accuracy, precision, recall, F1 and Matthews correlation coefficient (MCC) use threshold 0.5. Balanced accuracy weights the classes equally; precision asks how many predicted promoters are correct, recall asks how many true promoters are detected, and F1 balances both. MCC summarizes binary prediction/label association from −1 to 1, with 0 indicating no association. `zero_division=0` handles undefined precision/recall/F1 without hiding the poor prediction behavior.

ROC AUC and average precision (AP) summarize ranking across thresholds. ROC AUC of about 0.5 indicates random ranking; AP depends on the positive-class prevalence. **Log loss and Brier score are lower-is-better probability scores**. Log loss strongly penalizes confident mistakes; Brier is mean squared probability error. A good ranking does not guarantee well-calibrated probabilities.

The constant-probability reference uses the training promoter fraction, without fitting on validation labels. It contextualizes the metrics but is not a seventh trained model. Nominate the highest validation ROC AUC among the six paths before test evaluation. Ties follow the declared model order. This choice is used for paired comparisons later; do not replace it with the test winner.


In [ ]:
def calculate_metrics(labels, probabilities):
    predictions = probabilities >= threshold
    return {
        'accuracy': accuracy_score(labels, predictions),
        'balanced_accuracy': balanced_accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'f1': f1_score(labels, predictions, zero_division=0),
        'mcc': matthews_corrcoef(labels, predictions),
        'roc_auc': roc_auc_score(labels, probabilities),
        'average_precision': average_precision_score(labels, probabilities),
        'log_loss': log_loss(labels, probabilities),
        'brier_score': brier_score_loss(labels, probabilities),
    }

validation_probabilities = {}
for name in list(models)[:3]:
    counts = validation_counts.toarray() if name == 'k-mer + XGBoost' else validation_counts
    validation_probabilities[name] = models[name].predict_proba(counts)[:, 1]
del counts
validation_probabilities['Sequence CNN'] = predict_network(models['Sequence CNN'], raw_validation_loader)
validation_probabilities['Transformer + logistic regression'] = models['Transformer + logistic regression'].predict_proba(validation_means)[:, 1]
validation_probabilities['Transformer + CNN'] = predict_network(models['Transformer + CNN'], token_validation_loader)
validation_results = pd.DataFrame({name: calculate_metrics(y_validation, p)
                                   for name, p in validation_probabilities.items()}).T
validation_choice = validation_results['roc_auc'].idxmax()
print(validation_results.round(4).to_string())
print('Model nominated using validation ROC AUC:', validation_choice)
reference_validation = calculate_metrics(y_validation, np.full(len(y_validation), y_train.mean()))
print('Constant training-prevalence reference:', pd.Series(reference_validation).round(4).to_dict())


## 8. STOP before official test evaluation

Finish model choices now. The following cell downloads all 9,034 official test sequences, preserves the example containing `N`, and applies each existing transformation without refitting. Both CNNs use restored validation checkpoints; XGBoost uses its selected round. No test labels enter any fitting or selection step. **Run all includes this cell.**

Record representation time and classifier prediction time separately. A transformer model's complete prediction cost includes transformer inference even when its final logistic regression is very fast. Both frozen-transformer paths share extraction in this notebook; each standalone timing is charged the shared cost. Downstream prediction uses a full split in one call for classical models and mini-batches for neural models, so report seconds per 1,000 examples as throughput, not single-sequence latency.


In [ ]:
started = clock_time()
test_data = pd.read_parquet(data_url + 'test-00000-of-00001-d7614d9116c62f44.parquet')
test_data = test_data.rename(columns={'seq': 'sequence'})
test_data_download_seconds = clock_time() - started
assert test_data['sequence'].str.fullmatch('[ACGTN]{251}').all()
assert set(test_data['label']) == {0, 1}
y_test = test_data['label'].to_numpy()
started = clock_time()
test_counts = vectorizer.transform(test_data['sequence'])
kmer_test_seconds = clock_time() - started
test_probabilities = {}
test_head_seconds = {}
for name in list(models)[:3]:
    if name == 'k-mer + XGBoost':
        started = clock_time()
        counts = test_counts.toarray()
        xgb_test_dense_seconds = clock_time() - started
    else:
        counts = test_counts
    started = clock_time()
    test_probabilities[name] = models[name].predict_proba(counts)[:, 1]
    test_head_seconds[name] = clock_time() - started
del counts
started = clock_time()
test_onehot = encode_sequences(test_data['sequence'])
raw_test_seconds = clock_time() - started
raw_test_loader = DataLoader(TensorDataset(test_onehot, torch.tensor(y_test, dtype=torch.float32)),
                             batch_size=cnn_batch_size, shuffle=False)
started = clock_time()
test_probabilities['Sequence CNN'] = predict_network(models['Sequence CNN'], raw_test_loader)
test_head_seconds['Sequence CNN'] = clock_time() - started
started = clock_time()
test_means, test_tokens, test_mask = embed_sequences(test_data['sequence'])
transformer_test_seconds = clock_time() - started
token_test_loader = DataLoader(TensorDataset(test_tokens, test_mask, torch.tensor(y_test, dtype=torch.float32)),
                               batch_size=cnn_batch_size, shuffle=False)
name = 'Transformer + logistic regression'
started = clock_time()
test_probabilities[name] = models[name].predict_proba(test_means)[:, 1]
test_head_seconds[name] = clock_time() - started
name = 'Transformer + CNN'
started = clock_time()
test_probabilities[name] = predict_network(models[name], token_test_loader)
test_head_seconds[name] = clock_time() - started
test_results = pd.DataFrame({name: calculate_metrics(y_test, p)
                             for name, p in test_probabilities.items()}).T
reference_test = calculate_metrics(y_test, np.full(len(y_test), y_train.mean()))
print('Official test examples:', len(test_data), '| retained N examples:', test_data['sequence'].str.contains('N').sum())
print(test_results.round(4).to_string())
print('Constant training-prevalence reference:', pd.Series(reference_test).round(4).to_dict())


## 9. Ranking, classification, and calibration

ROC and precision–recall overlays compare ranking across thresholds. Dashed lines show random ranking and the actual split's positive fraction. Confusion matrices at 0.5 show the type and count of mistakes, with identical color limits across models. Rows are actual labels; columns are predictions, ordered non-promoter then promoter.

A reliability diagram compares average predicted probability with the observed promoter fraction. `n_bins=10, strategy='quantile'` forms groups with roughly equal example counts, reducing empty bins. The diagonal represents ideal calibration. Bins still have sampling uncertainty, especially at extremes. These diagrams diagnose the existing probabilities; no test-based calibration or threshold changes are performed. The same plots are generated for validation and test.


In [ ]:
def plot_comparison(labels, probabilities, split):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for name, p in probabilities.items():
        RocCurveDisplay.from_predictions(labels, p, ax=axes[0], name=name)
        PrecisionRecallDisplay.from_predictions(labels, p, ax=axes[1], name=name)
        observed, predicted = calibration_curve(labels, p, n_bins=10, strategy='quantile')
        axes[2].plot(predicted, observed, marker='o', label=name)
    axes[0].plot([0, 1], [0, 1], 'k--')
    axes[1].axhline(labels.mean(), color='black', linestyle='--')
    axes[2].plot([0, 1], [0, 1], 'k--')
    axes[2].set(xlabel='Mean predicted probability', ylabel='Observed promoter fraction', title='Calibration')
    axes[2].legend(fontsize=7)
    fig.suptitle(split.capitalize() + ': ranking and calibration')
    fig.tight_layout()
    fig.savefig(output_dir / (split + '_curves.png'), dpi=150)
    plt.show()
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, (name, p) in zip(axes.flat, probabilities.items()):
        ConfusionMatrixDisplay.from_predictions(
            labels, p >= threshold, labels=[0, 1], display_labels=['Non-promoter', 'Promoter'],
            ax=ax, colorbar=False, cmap='Blues', im_kw={'vmin': 0, 'vmax': np.bincount(labels).max()},
        )
        ax.set_title(name, fontsize=10)
    fig.suptitle(split.capitalize() + ': confusion matrices at threshold 0.5')
    fig.tight_layout()
    fig.savefig(output_dir / (split + '_confusion.png'), dpi=150)
    plt.show()

plot_comparison(y_validation, validation_probabilities, 'validation')
plot_comparison(y_test, test_probabilities, 'test')


## 10. Paired uncertainty and shared errors

Small score differences may reflect sampling variation. Stratified bootstrap samples test indices with replacement separately within each label, preserving class counts. **Use the same resampled indices for every model** so differences are paired. Report percentile 95% intervals (2.5th to 97.5th percentile) for ROC AUC and AP, and a ROC AUC difference relative to the model nominated on validation. A paired difference interval containing zero does not provide clear evidence of a difference under this resampling scheme. Overlap of separate score intervals is not itself a paired test.

These intervals condition on one fitted model and one split; they do not measure variation across training seeds or chromosomes. Resampling sequences treats them as independent, which may be optimistic when genomic examples are related. AP intervals also condition on the benchmark's class proportions. The code reports uncertainty without changing any fitted model.

The disagreement heatmap shows the fraction of examples on which two thresholded predictions differ. Low disagreement means similar decisions, not necessarily high accuracy. Export each sequence's number of wrong predictions, GC fraction, and presence of `N` for inspection. A mistake shared by all models can reflect difficult biology, representation limits or benchmark issues; it is not proof of a mislabeled example.


In [ ]:
rng = np.random.default_rng(seed)
class_indices = [np.flatnonzero(y_test == label) for label in [0, 1]]
bootstrap = {name: np.empty((bootstrap_repeats, 2)) for name in models}
for repeat in range(bootstrap_repeats):
    indices = np.concatenate([rng.choice(rows, size=len(rows), replace=True) for rows in class_indices])
    for name, p in test_probabilities.items():
        bootstrap[name][repeat] = [roc_auc_score(y_test[indices], p[indices]),
                                   average_precision_score(y_test[indices], p[indices])]
    if (repeat + 1) % 100 == 0:
        print('Bootstrap samples:', repeat + 1, flush=True)
interval_rows = []
for name, values in bootstrap.items():
    lower, upper = np.quantile(values, [0.025, 0.975], axis=0)
    differences = values[:, 0] - bootstrap[validation_choice][:, 0]
    delta_lower, delta_upper = np.quantile(differences, [0.025, 0.975])
    interval_rows.append({
        'model': name, 'roc_auc_lower': lower[0], 'roc_auc_upper': upper[0],
        'ap_lower': lower[1], 'ap_upper': upper[1],
        'delta_auc_vs_validation_choice': test_results.loc[name, 'roc_auc'] - test_results.loc[validation_choice, 'roc_auc'],
        'delta_auc_lower': delta_lower, 'delta_auc_upper': delta_upper,
    })
confidence_intervals = pd.DataFrame(interval_rows).set_index('model')
print('Paired reference (chosen on validation):', validation_choice)
print(confidence_intervals.round(4).to_string())
decisions = np.column_stack([p >= threshold for p in test_probabilities.values()])
disagreement = (decisions[:, :, None] != decisions[:, None, :]).mean(axis=0)
fig, ax = plt.subplots(figsize=(9, 7))
heatmap = ax.imshow(disagreement, vmin=0, vmax=1, cmap='Blues')
ax.set_xticks(range(len(models)), models, rotation=45, ha='right')
ax.set_yticks(range(len(models)), models)
for row in range(len(models)):
    for col in range(len(models)):
        ax.text(col, row, f'{disagreement[row, col]:.3f}', ha='center', va='center')
fig.colorbar(heatmap, ax=ax, label='Fraction of different class predictions')
ax.set_title('Test prediction disagreement')
fig.tight_layout()
fig.savefig(output_dir / 'test_disagreement.png', dpi=150)
plt.show()
error_analysis = pd.DataFrame({
    'sample_id': 'test_' + test_data.index.astype(str), 'label': y_test,
    'wrong_models': (decisions != y_test[:, None]).sum(axis=1),
    'gc_fraction': test_data['sequence'].str.count('[GC]').to_numpy() / 251,
    'contains_N': test_data['sequence'].str.contains('N').to_numpy(),
})
print('Test examples by number of wrong model predictions:')
print(error_analysis['wrong_models'].value_counts().sort_index())


## 11. Performance versus computation and model size

Separate training/validation feature preparation, downstream fitting (including checkpoint selection), test feature preparation and test classifier prediction. `fit_with_features_seconds` adds training and validation feature costs to fitting; `test_total_seconds` adds test representation and prediction. Downloads, model loading, plotting, bootstrap, and artifact serialization are excluded. Transformer timings include tokenization and keeping the token cache in RAM for both paths, even when only mean features are needed: this shared benchmark implementation overestimates a specialized mean-only deployment.

`classifier_file_mib` measures a serialized downstream model, not peak RAM or complete deployment size. K-mer models also need the shared vocabulary; transformer models also need the shared pretrained weights and tokenizer. `learned_coefficients` counts linear/CNN parameters and is undefined for trees; `tree_nodes` describes tree size separately. A tree node is not equivalent to a neural-network weight. `frozen_parameters` records the transformer's substantial extra capacity. Model state files need the architectures declared above to reload.

The scatter plot uses logarithmic end-to-end test seconds per 1,000 sequences versus test ROC AUC with bootstrap intervals. A desirable trade-off combines strong predictive performance and acceptable compute, rather than optimizing a single score. CPU/GPU differences, batching, the single training seed and the absence of repeated timing trials limit generalization of the resource comparison.


In [ ]:
representation_times = {
    'k-mer + logistic regression': (kmer_train_seconds, kmer_validation_seconds, kmer_test_seconds),
    'k-mer + random forest': (kmer_train_seconds, kmer_validation_seconds, kmer_test_seconds),
    'k-mer + XGBoost': (kmer_train_seconds + xgb_train_dense_seconds,
                        kmer_validation_seconds + xgb_validation_dense_seconds, kmer_test_seconds + xgb_test_dense_seconds),
    'Sequence CNN': (raw_train_seconds, raw_validation_seconds, raw_test_seconds),
    'Transformer + logistic regression': (transformer_train_seconds, transformer_validation_seconds, transformer_test_seconds),
    'Transformer + CNN': (transformer_train_seconds, transformer_validation_seconds, transformer_test_seconds),
}
frozen_parameters = sum(p.numel() for p in foundation_model.parameters())
frozen_weights_mib = sum(p.numel() * p.element_size() for p in foundation_model.parameters()) / 2**20
resource_rows = []
for number, (name, model) in enumerate(models.items(), start=1):
    path = output_dir / f'model_{number}.pt' if isinstance(model, nn.Module) else output_dir / f'model_{number}.joblib'
    coefficients, tree_nodes = np.nan, np.nan
    if isinstance(model, nn.Module):
        torch.save({key: value.cpu() for key, value in model.state_dict().items()}, path)
        coefficients = sum(p.numel() for p in model.parameters())
    else:
        joblib.dump(model, path)
        if name == 'k-mer + random forest':
            tree_nodes = sum(tree.tree_.node_count for tree in model.estimators_)
        elif name == 'k-mer + XGBoost':
            tree_nodes = len(model.get_booster().trees_to_dataframe())
        else:
            linear = model[-1] if name.startswith('Transformer') else model
            coefficients = linear.coef_.size + linear.intercept_.size
    train_rep, validation_rep, test_rep = representation_times[name]
    resource_rows.append({
        'model': name, 'train_features_seconds': train_rep, 'validation_features_seconds': validation_rep,
        'fit_seconds': fit_seconds[name], 'fit_with_features_seconds': train_rep + validation_rep + fit_seconds[name],
        'test_features_seconds': test_rep, 'test_classifier_seconds': test_head_seconds[name],
        'test_total_seconds': test_rep + test_head_seconds[name],
        'test_seconds_per_1000': 1000 * (test_rep + test_head_seconds[name]) / len(y_test),
        'classifier_file_mib': path.stat().st_size / 2**20, 'learned_coefficients': coefficients,
        'tree_nodes': tree_nodes, 'frozen_parameters': frozen_parameters if name.startswith('Transformer') else 0,
        'selected_epoch_or_round': selected_steps.get(name, np.nan), 'artifact': path.name,
    })
resources = pd.DataFrame(resource_rows).set_index('model')
joblib.dump(vectorizer, output_dir / 'kmer_vectorizer.joblib')
print(resources.round(3).to_string())
print('Additional shared pretrained weight storage (MiB):', round(frozen_weights_mib, 2))
print('Pretrained model load/download seconds (excluded above):', round(model_load_seconds, 2))
fig, ax = plt.subplots(figsize=(11, 6))
for number, name in enumerate(models, start=1):
    score = test_results.loc[name, 'roc_auc']
    low, high = confidence_intervals.loc[name, ['roc_auc_lower', 'roc_auc_upper']]
    x = resources.loc[name, 'test_seconds_per_1000']
    ax.errorbar(x, score, yerr=[[max(0, score - low)], [max(0, high - score)]], fmt='o', capsize=4, label=f'{number}. {name}')
    ax.annotate(str(number), (x, score), xytext=(5, 7), textcoords='offset points')
ax.set_xscale('log')
ax.set(xlabel='End-to-end test seconds per 1,000 sequences (log scale)', ylabel='Test ROC AUC',
       title='Predictive performance versus computation (95% bootstrap intervals)')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(output_dir / 'performance_vs_compute.png', dpi=150)
plt.show()


## 12. Export and discuss the benchmark

Save full-precision metrics, uncertainty intervals, resource measurements, fitted downstream artifacts, row-aligned probabilities, neural learning curves and a run manifest. The manifest records settings, data/model revisions, versions, hardware, split sizes and the validation-based choice. `sample_id` uses the original official file row. The model-number artifact order follows the six paths in the introduction. Run this cell to finalize the outputs. Token embeddings remain in notebook memory rather than being exported to disk.

Use the results to answer:

1. Does the validation-nominated model remain strong on test? Do ROC AUC, AP, MCC, log loss and calibration tell the same story?
2. Do the paired difference intervals distinguish models, or are some differences too small for this test set? How would multiple training seeds change the study?
3. Which gains come with a large feature-extraction cost? How does the answer change for a cached dataset versus new sequences arriving in production?
4. Do the two transformer heads make similar mistakes? Does their comparison isolate convolution, or also pooling, capacity and optimization?
5. Would a chromosome-held-out split, related-sequence filtering or a new species produce the same ranking? Pretraining may contain related genomes, so do not claim that a test win establishes a new biological mechanism or generalization to unseen species.

A useful conclusion states the dataset, split, settings, uncertainty and computation budget supporting it. No model should be declared universally best from this single run.


In [ ]:
validation_results.to_csv(output_dir / 'validation_metrics.csv')
test_results.to_csv(output_dir / 'test_metrics.csv')
confidence_intervals.to_csv(output_dir / 'test_confidence_intervals.csv')
resources.to_csv(output_dir / 'resources.csv')
error_analysis.to_csv(output_dir / 'test_errors.csv', index=False)
pd.DataFrame(disagreement, index=models, columns=models).to_csv(output_dir / 'test_disagreement.csv')
for split, data, probabilities in [('validation', validation_data, validation_probabilities),
                                    ('test', test_data, test_probabilities)]:
    predictions = pd.DataFrame({'sample_id': split + '_' + data.index.astype(str),
                                'label': data['label'].to_numpy(), **probabilities})
    predictions.to_csv(output_dir / (split + '_predictions.csv'), index=False)
for number, (name, history) in enumerate(histories.items(), start=1):
    history.to_csv(output_dir / f'cnn_{number}_learning_curve.csv', index=False)
manifest = {
    'seed': seed, 'k': k, 'threshold': threshold, 'max_epochs': max_epochs,
    'learning_rate': learning_rate, 'batch_size': batch_size, 'cnn_batch_size': cnn_batch_size,
    'bootstrap_repeats': bootstrap_repeats, 'cpu_threads': n_jobs,
    'dataset_url': data_url, 'model_name': model_name, 'model_revision': model_revision,
    'split_sizes': {'train': len(train_data), 'validation': len(validation_data), 'test': len(test_data)},
    'validation_choice_by_roc_auc': validation_choice, 'selected_steps': selected_steps,
    'versions': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                 'sklearn': sklearn.__version__, 'torch': torch.__version__,
                 'transformers': transformers.__version__, 'xgboost': xgboost.__version__},
    'hardware': {'platform': platform.platform(), 'cpu': platform.processor(), 'logical_cpus': os.cpu_count(),
                 'device': str(device), 'gpu': torch.cuda.get_device_name() if device.type == 'cuda' else None},
    'model_load_seconds': model_load_seconds, 'frozen_weight_mib': frozen_weights_mib,
    'training_data_download_seconds': training_data_download_seconds, 'test_data_download_seconds': test_data_download_seconds,
    'reference_validation': reference_validation, 'reference_test': reference_test,
}
(output_dir / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
print('Saved benchmark outputs to:', output_dir.resolve())
